In [16]:
import os

# ==========================================
# 强制注入代理设置 (根据你本地转发的端口修改，通常是 7890)
proxy_url = "http://127.0.0.1:7890" 

os.environ["http_proxy"] = proxy_url
os.environ["https_proxy"] = proxy_url
os.environ["HTTP_PROXY"] = proxy_url
os.environ["HTTPS_PROXY"] = proxy_url
# ==========================================

In [ ]:
import sys
from rdkit import Chem

# smiles规范化
def get_canonical(smi):
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                can_smi = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
                print(f"规范化结果: {can_smi}")
            else:
                print("错误: 无法解析该 SMILES，请检查输入是否正确。")
        except Exception as e:
            print(f"程序运行出错: {e}")

if __name__ == "__main__":
    smi = "[Cl].CC(C)NCC(O)COc1cccc2ccccc12"
    smi1 = "CC(C)NCC(COC1=CC=CC2=CC=CC=C21)O.Cl"
    get_canonical(smi)

规范化结果: CC(C)NCC(O)COc1cccc2ccccc12.[Cl]


In [ ]:
import sys
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments, AllChem
import pubchempy as pcp

def get_structural_features(mol):
    """使用 RDKit 提取关键官能团描述"""
    features = []
    # 环系统
    ring_info = mol.GetRingInfo()
    if ring_info.NumRings() > 0:
        features.append(f"{ring_info.NumRings()} ring system(s)")
    
    # 常见官能团检测
    if Fragments.fr_benzene(mol) > 0: features.append("aromatic benzene rings")
    if Fragments.fr_amide(mol) > 0: features.append("amide linkages")
    if Fragments.fr_NH2(mol) > 0: features.append("primary amine groups")
    if Fragments.fr_Ar_OH(mol) > 0: features.append("phenolic hydroxyl groups")
    if Fragments.fr_ester(mol) > 0: features.append("ester functional groups")
    if Fragments.fr_halogen(mol) > 0: features.append("halogen substituents")
    
    if not features:
        return "a complex aliphatic structure"
    return ", ".join(features)

def generate_expert_description(smi):
    """模仿 Uni-Poly 作者口吻生成学术描述"""
    mol = Chem.MolFromSmiles(smi)
    if not mol:
        return "Error: Invalid SMILES"

    # 1. 物理化学参数计算
    mw = round(Descriptors.MolWt(mol), 2)
    logp = round(Descriptors.MolLogP(mol), 2)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    formula = Chem.rdMolDescriptors.CalcMolFormula(mol)
    tpsa = round(Descriptors.TPSA(mol), 2)

    # 2. 提取结构特征
    feat_text = get_structural_features(mol)

    # 3. 拼凑 Uni-Poly 风格的段落
    description = (
        f"This molecule, with the chemical formula {formula}, has a molecular weight of {mw} g/mol. "
        f"Its structural architecture is characterized by {feat_text}. "
        f"From a physicochemical perspective, it exhibits a calculated LogP of {logp} and a Total Polar Surface Area (TPSA) of {tpsa} Å², "
        f"indicating specific lipophilic and membrane permeability characteristics. "
        f"The presence of {hbd} hydrogen bond donor(s) and {hba} acceptor(s) further facilitates its potential "
        f"interaction within biological targets or synthetic environments. "
        f"These attributes make it a significant subject for property prediction and molecular representation studies."
    )
    return description



In [ ]:
import requests  # 导入网络请求库
import os        # 导入系统库用于代理设置
import pubchempy as pcp  # 导入 pubchempy 用于获取 CID

# ==========================================
# 1. 代理设置 (确保 Python 进程能联网)
# ==========================================
proxy_url = "http://127.0.0.1:7890"  # 本地代理端口
os.environ["http_proxy"] = proxy_url
os.environ["https_proxy"] = proxy_url

def get_full_wikipedia_description(smiles):
    """
    专门提取 PubChem 网页顶部的 'Record Description' 模块
    """
    try:
        # 第一步：通过 SMILES 转换拿到 CID
        compounds = pcp.get_compounds(smiles, namespace='smiles')
        if not compounds: return "CID not found."
        cid = compounds[0].cid
        
        # 第二步：访问 PUG View 接口，这个接口的数据结构与网页显示完全对应
        pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON"
        response = requests.get(pug_url, timeout=15)
        data = response.json()
        # print(data)

        # 第三步：精准寻找 "Record Description" 这一节
        # 我们只从特定的 TOCHeading 中提取文字内容
        def find_record_description_section(sections):
            text_parts = []
            for sec in sections:
                # 网页上那段话通常位于 TOCHeading 为 'Record Description' 的模块下
                if sec.get('TOCHeading') == 'Record Description':
                    # 提取该节下所有的文字信息
                    info_list = sec.get('Information', [])
                    for info in info_list:
                        # 查找 Value 里的字符串
                        val = info.get('Value', {})
                        for markup in val.get('StringWithMarkup', []):
                            s = markup.get('String', '')
                            if s: text_parts.append(s)
                
                # 如果当前层没找到，递归进入子节寻找
                if 'Section' in sec:
                    sub_parts = find_record_description_section(sec['Section )
                    text_parts.extend(sub_parts)
            return text_parts

        # 执行搜索逻辑
        all_descriptions = []
        if 'Record' in data and 'Section' in data['Record :
            all_descriptions = find_record_description_section(data['Record ['Section )

        # 第四步：清洗和合并
        # 去掉重复的段落并用换行符连接
        clean_descriptions = []
        for d in all_descriptions:
            # 过滤掉过短的无意义字符或版权标识
            if len(d) > 30 and "Copyright" not in d:
                clean_descriptions.append(d.strip())
        
        # 移除完全重复的段落
        final_list = list(dict.fromkeys(clean_descriptions))
        
        # 将所有百科段落拼成一个完整的文本块
        return "\n\n".join(final_list)

    except Exception as e:
        return f"Error during fetching: {str(e)}"

# ==========================================
# 2. 测试运行 (以酒石酸为例)
# ==========================================
test_smi = "C1=CC=CC=C1CC(N2CCCC2)CCC"  # 截图中的分子 SMILES
print(f"正在抓取分子的完整百科描述...\n")

full_text = get_full_wikipedia_description(test_smi)

print("="*60)
print(full_text)
print("="*60)

正在抓取分子的完整百科描述...

Prolintane is a member of amphetamines.

PROLINTANE is a small molecule drug with a maximum clinical trial phase of II and has 1 investigational indication.

See also: Prolintane Hydrochloride (active moiety of).


In [47]:
import requests
import os
import pubchempy as pcp

# ==========================================
# 1. 代理设置 (请确保端口与你本地一致)
# ==========================================
proxy_url = "http://127.0.0.1:7890" 
os.environ["http_proxy"] = proxy_url
os.environ["https_proxy"] = proxy_url

def get_optimized_description(smiles):
    """
    根据用户提供的 JSON 结构，精准提取 Names and Identifiers 下的 Record Description
    """
    try:
        # 第一步：获取 CID
        compounds = pcp.get_compounds(smiles, namespace='smiles')
        if not compounds: return "CID not found."
        cid = compounds[0].cid
        
        # 第二步：获取 PUG View JSON
        pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON"
        response = requests.get(pug_url, timeout=15)
        if response.status_code != 200: return "API Error."
        
        data = response.json()
        final_descriptions = []

        # 第三步：按照目标路径精准导航
        # 路径: Record -> Section
        sections = data.get("Record", {}).get("Section", [])
        
        for sec in sections:
            # 找到一级目录: Names and Identifiers
            if sec.get("TOCHeading") == "Names and Identifiers":
                sub_sections = sec.get("Section", [])
                
                for sub_sec in sub_sections:
                    # 找到二级目录: Record Description
                    if sub_sec.get("TOCHeading") == "Record Description":
                        informations = sub_sec.get("Information", [])
                        
                        for info in informations:
                            # 提取 StringWithMarkup 里的所有文本
                            value = info.get("Value", {})
                            markup_list = value.get("StringWithMarkup", [])
                            
                            for markup in markup_list:
                                text = markup.get("String", "")
                                if text:
                                    final_descriptions.append(text.strip())

        # 第四步：拼接所有找到的段落
        if not final_descriptions:
            return "No Record Description found for this CID."
            
        return "\n\n".join(final_descriptions)

    except Exception as e:
        return f"Error: {str(e)}"

# ==========================================
# 2. 测试运行
# ==========================================
test_smi = "C(C(C(=O)O)O)(C(=O)O)O" 
print("正在执行精准路径提取...\n")
result = get_optimized_description(test_smi)

print("="*60)
print(result)
print("="*60)

正在执行精准路径提取...

2,3-dihydroxybutanedioic acid is a tetraric acid that is butanedioic acid substituted by hydroxy groups at positions 2 and 3. It has a role as a human xenobiotic metabolite and a plant metabolite. It is a conjugate acid of a 3-carboxy-2,3-dihydroxypropanoate.

Tartaric acid has been reported in Camellia sinensis, Catunaregam spinosa, and other organisms with data available.

Tartaric acid is a white crystalline organic acid. It occurs naturally in many plants, particularly grapes and tamarinds, and is one of the main acids found in wine. It is added to other foods to give a sour taste, and is used as an antioxidant. Salts of tartaric acid are known as tartrates. It is a dihydroxy derivative of dicarboxylic acid. Tartaric acid is a muscle toxin, which works by inhibiting the production of malic acid, and in high doses causes paralysis and death. The minimum recorded fatal dose for a human is about 12 grams. In spite of that, it is included in many foods, especially sour-t

In [57]:
import sys  # 导入系统库，用于读取命令行参数
from rdkit import Chem  # 导入 RDKit 核心功能模块
from rdkit.Chem import Descriptors, rdMolDescriptors, Fragments  # 导入描述符、高级描述符和官能团识别模块

def extract_chemical_knowledge(smiles):  # 定义核心函数：输入 SMILES，输出学术描述
    # 1. 分子对象创建与合法性检查
    mol = Chem.MolFromSmiles(smiles)  # 将输入的字符串转换为 RDKit 分子对象
    if not mol:  # 如果转换失败（SMILES 格式错误）
        return "Error: Invalid SMILES string."  # 返回错误提示并终止

    # 为了确保输出的描述是针对标准结构的，进行规范化处理
    canonical_smi = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)  # 生成标准且含手性的 SMILES
    mol = Chem.MolFromSmiles(canonical_smi)  # 重新加载标准分子对象以保证计算精度

    # 2. 基础物理常数提取
    formula = rdMolDescriptors.CalcMolFormula(mol)  # 计算分子的化学式（如 C27H33N3O2）
    MolWt = round(Descriptors.MolWt(mol), 2)  # 计算精确分子量，保留两位小数
    logp = round(Descriptors.MolLogP(mol), 2)  # 计算脂水分配系数 LogP（反映亲脂性）
    tpsa = round(Descriptors.TPSA(mol), 2)  # 计算总极性表面积 TPSA（反映渗透性）

    # 3. 结构特征与原子统计
    hbd = Descriptors.NumHDonors(mol)  # 统计氢键供体数量
    hba = Descriptors.NumHAcceptors(mol)  # 统计氢键受体数量
    rings = mol.GetRingInfo().NumRings()  # 统计分子中环的总数
    rot_bonds = Descriptors.NumRotatableBonds(mol)  # 统计可旋转化学键数量（反映分子柔性）
    stereo_centers = rdMolDescriptors.CalcNumAtomStereoCenters(mol)  # 统计手性中心（立体中心）的数量
    heavy_atoms = mol.GetNumHeavyAtoms()

    # 4. 关键官能团扫描（使用 RDKit 内置的片段库）
    fg_list = []  # 创建一个列表用于存放检测到的官能团
    if Fragments.fr_benzene(mol) > 0: fg_list.append("aromatic benzene rings")  # 检测苯环
    if Fragments.fr_amide(mol) > 0: fg_list.append("amide groups")  # 检测酰胺键
    if Fragments.fr_Ar_OH(mol) > 0: fg_list.append("phenolic hydroxyls")  # 检测酚羟基
    if Fragments.fr_NH2(mol) > 0: fg_list.append("primary amine groups")  # 检测伯胺
    if Fragments.fr_halogen(mol) > 0: fg_list.append("halogen substituents")  # 检测卤素（F, Cl, Br, I）
    if Fragments.fr_ester(mol) > 0: fg_list.append("ester linkages")  # 检测酯基
    if Fragments.fr_ether(mol) > 0: fg_list.append("ether groups")  # 检测醚键
    
    # 构造官能团描述短语
    if fg_list:  # 如果找到了已知官能团
        fg_text = "The structural framework incorporates " + ", ".join(fg_list) + ". "  # 拼接官能团描述词
    else:  # 如果没找到常见官能团
        fg_text = "The structure consists of a complex hydrocarbon-based framework. "  # 使用通用描述

    # 5. 按照 Uni-Poly 风格拼凑学术段落
    description = (
        f"This compound, defined by the chemical formula {formula}, possesses a molecular weight of {MolWt} g/mol. "
        f"{fg_text}"
        f"Topological analysis indicates the presence of {rings} ring system(s) and {stereo_centers} stereogenic center(s), "
        f"From a physicochemical perspective, the molecule exhibits a LogP of {logp} and a TPSA of {tpsa} Å². "
        f"It contains {heavy_atoms} heavy atom(s), reflecting the size of its non-hydrogen atomic framework. "
        f"The molecular flexibility is moderated by {rot_bonds} rotatable bonds, while its interaction profile is defined by "
        f"{hbd} hydrogen bond donor(s) and {hba} acceptor(s)."
    )



    return canonical_smi, description  # 返回规范化 SMILES 和最终生成的知识文本

if __name__ == "__main__":  # 如果脚本被直接运行
    input_smi = "C(C(C(=O)O)O)(C(=O)O)O"
    # 执行提取逻辑
    can_smi, text_info = extract_chemical_knowledge(input_smi)
    
    # 打印结果
    print("\n" + "="*80)  # 顶部装饰线
    print(f"Standardized Identity (SMILES): {can_smi}")  # 打印处理后的唯一标识
    print("-" * 80)  # 分隔线
    print(f"Expert Textual Knowledge (RDKit-based):\n{text_info}")  # 打印最终生成的知识文本
    print("="*80 + "\n")  # 底部装饰线


Standardized Identity (SMILES): O=C(O)C(O)C(O)C(=O)O
--------------------------------------------------------------------------------
Expert Textual Knowledge (RDKit-based):
This compound, defined by the chemical formula C4H6O6, possesses a molecular weight of 150.09 g/mol. The structure consists of a complex hydrocarbon-based framework. Topological analysis indicates the presence of 0 ring system(s) and 2 stereogenic center(s), From a physicochemical perspective, the molecule exhibits a LogP of -2.12 and a TPSA of 115.06 Å². It contains 10 heavy atom(s), reflecting the size of its non-hydrogen atomic framework. The molecular flexibility is moderated by 3 rotatable bonds, while its interaction profile is defined by 4 hydrogen bond donor(s) and 4 acceptor(s).



In [60]:
# 纯净的理化性质
def get_all_rdkit_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    # 1. 基础识别
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mw = round(Descriptors.MolWt(mol), 4)
    exact_mw = round(Descriptors.ExactMolWt(mol), 4)
    
    # 2. 理化性质
    logp = round(Descriptors.MolLogP(mol), 4)
    tpsa = round(Descriptors.TPSA(mol), 4)
    
    # 3. 结构计数
    heavy_atoms = mol.GetNumHeavyAtoms()
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    rot_bonds = Descriptors.NumRotatableBonds(mol)
    
    # 4. 环与拓扑
    rings = rdMolDescriptors.CalcNumRings(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    stereo_centers = rdMolDescriptors.CalcNumAtomStereoCenters(mol)

    return {
        "Formula": formula, "MolWt": mw, "ExactMolWt": exact_mw,
        "LogP": logp, "TPSA": tpsa, "Heavy_Atoms": heavy_atoms,
        "H_Bond_Donors": hbd, "H_Bond_Acceptors": hba, "Rotatable_Bonds": rot_bonds,
        "Ring_Count": rings, "Aromatic_Rings": aromatic_rings,
        "Stereocenters": stereo_centers
    }

In [61]:
print(get_all_rdkit_properties("C(C(C(=O)O)O)(C(=O)O)O"))

{'Formula': 'C4H6O6', 'MolWt': 150.086, 'ExactMolWt': 150.0164, 'LogP': -2.1226, 'TPSA': 115.06, 'Heavy_Atoms': 10, 'H_Bond_Donors': 4, 'H_Bond_Acceptors': 4, 'Rotatable_Bonds': 3, 'Ring_Count': 0, 'Aromatic_Rings': 0, 'Stereocenters': 2}
